### **Part A**

[Embedded submission image omitted from public copy.]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


df = pd.read_csv('hotel_bookings.csv')
selected_columns = ['hotel', 'lead_time', 'arrival_date_month', 'adults', 'children', 'babies', 'is_canceled', 'adr', 'total_of_special_requests', 'stays_in_week_nights', 'stays_in_weekend_nights']
df_selected = df[selected_columns]
df_sampled = df_selected.sample(n=10000, random_state=7)

selected_columns=['hotel', 'lead_time', 'is_canceled', 'adr']
df_sampled = df_sampled[selected_columns]
df_sampled_200 = df_sampled.sample(n=200, random_state=7)

In [ ]:
numerical_columns_with_limits = {
    'adr': 1000,
    'lead_time': 1000,
}

for col, max_value in numerical_columns_with_limits.items():
    if col in ['adr', 'lead_time']:
        plt.figure(figsize=(8, 5))
        sns.histplot(df_sampled_200[df_sampled_200[col] <= max_value][col], kde=False, bins=15, color='blue')
        plt.title(f'Histogram of {col} ')
        plt.xlabel(col)
        plt.ylabel('Frequency')
        plt.show()

        plt.figure(figsize=(8, 5))
        sns.boxplot(x=df_sampled_200[df_sampled_200[col] <= max_value][col], color='skyblue')
        plt.title(f'Box Plot of {col} ')
        plt.xlabel(col)
        plt.show()

categorical_columns = ['hotel', 'is_canceled']

for col in categorical_columns:
    plt.figure(figsize=(10, 6))
    sns.countplot(data=df_sampled_200, x=col, palette='pastel')
    plt.title(f'Bar Plot of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.show()

**Q 1.1**

In [ ]:
import statsmodels.api as sm
import pandas as pd
from scipy import stats
from scipy.stats import norm

df_sampled_encoded_200 = df_sampled_200.copy()
df_sampled_encoded_200['hotel'] = df_sampled_200['hotel'].map({'City Hotel': 0, 'Resort Hotel': 1})

X = df_sampled_encoded_200[['hotel', 'is_canceled', 'adr']]
X = sm.add_constant(X)  # Add intercept
y = df_sampled_encoded_200['lead_time']

model = sm.OLS(y, X).fit()
stderr_200 = model.bse.iloc[0]

beta_hat_200 = model.params
beta_hat_200 = beta_hat_200.to_numpy()

se_beta = model.bse

alpha = 0.05
z_value = norm.ppf(1 - alpha / 2)

confidence_intervals = pd.DataFrame({
    'Coefficient': model.params,
    'Std Error': se_beta,
    'Lower Bound': model.params - z_value * se_beta,
    'Upper Bound': model.params + z_value * se_beta
})

ci_list_q_1 = []
ci_length_1 = []
# Iterate over each feature and store its confidence interval
for i, feature in enumerate(['Intercept', 'hotel', 'is_canceled', 'adr']):
    lower_bound = confidence_intervals['Lower Bound'].iloc[i]
    upper_bound = confidence_intervals['Upper Bound'].iloc[i]
    ci_length_1.append(upper_bound - lower_bound)
    print(f"{feature}: ({lower_bound:.4f}, {upper_bound:.4f})")
    ci_list_q_1.append((lower_bound, upper_bound))


Calc Betas of bootstarp

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# Assuming df_sampled_encoded_200 is already defined and contains the relevant columns
df_sampled_encoded_200 = df_sampled_200.copy()
df_sampled_encoded_200['hotel'] = df_sampled_200['hotel'].map({'City Hotel': 0, 'Resort Hotel': 1})

B = 400  # Number of bootstrap samples

betas = []

for i in range(B):
    sample = df_sampled_encoded_200.sample(n=len(df_sampled_encoded_200), replace=True)
    X = sample[['hotel', 'is_canceled', 'adr']]
    X = sm.add_constant(X)  # Add constant for the intercept term
    y = sample['lead_time']  # Replace 'target_column' with the actual dependent variable name
    model = sm.OLS(y, X).fit()
    betas.append(model.params)

# Convert betas to a NumPy array for easier manipulation
betas = np.array(betas)
beta_means = np.mean(betas, axis=0)
beta_variances = np.var(betas, axis=0)

**Q 1.2**

In [ ]:
z_alpha_2 = norm.ppf(1 - 0.05 / 2)  # For 95% CI, use z value for 0.025 in each tail
beta_standard_errors = np.sqrt(beta_variances)
lower_bound = beta_means - z_alpha_2 * beta_standard_errors
upper_bound = beta_means + z_alpha_2 * beta_standard_errors

ci_length_2 = []

for i, feature in enumerate(['Intercept', 'hotel', 'is_canceled', 'adr']):
    print(f"{feature}: ({lower_bound[i]:.4f}, {upper_bound[i]:.4f})")
    ci_length_2.append(upper_bound[i] - lower_bound[i])
    # Save the confidence interval as a tuple in the list
    ci_list_q_1.append((lower_bound[i], upper_bound[i]))

**Q 1.3**

In [ ]:
# Assuming df_sampled_encoded_200 is already defined and contains the relevant columns
df_sampled_encoded_200 = df_sampled_200.copy()
df_sampled_encoded_200['hotel'] = df_sampled_200['hotel'].map({'City Hotel': 0, 'Resort Hotel': 1})

alpha = 0.05  # For 95% confidence intervals

# Fit the model to the original data to get θ₀
X_original = df_sampled_encoded_200[['hotel', 'is_canceled', 'adr']]
X_original = sm.add_constant(X_original)
y_original = df_sampled_encoded_200['lead_time']
original_model = sm.OLS(y_original, X_original).fit()
theta_0 = original_model.params.values  # Convert to numpy array

# Calculate the bootstrap estimates minus the original estimate (√n(θ̂ₙ - θ₀))
n = len(df_sampled_encoded_200)
scaled_differences = np.sqrt(n) * (betas - theta_0[np.newaxis, :])

# Calculate the quantiles for each parameter
lower_quantiles = np.percentile(scaled_differences, alpha/2 * 100, axis=0)
upper_quantiles = np.percentile(scaled_differences, (1-alpha/2) * 100, axis=0)

# Calculate the pivotal confidence intervals
lower_bounds = theta_0 - upper_quantiles/np.sqrt(n)
upper_bounds = theta_0 - lower_quantiles/np.sqrt(n)

# Create a DataFrame with the results
feature_names = ['const', 'hotel', 'is_canceled', 'adr']
results_df = pd.DataFrame({
    'Feature': feature_names,
    'Original Estimate': theta_0,
    'Lower Bound (95%)': lower_bounds,
    'Upper Bound (95%)': upper_bounds
})

ci_length_3=[]

for i, feature in enumerate(feature_names):
    ci_length_3.append(upper_bounds[i] - lower_bounds[i])
    print(f"{feature}: ({lower_bounds[i]:.4f}, {upper_bounds[i]:.4f})")
    ci_list_q_1.append((lower_bounds[i], upper_bounds[i]))

**Q 1.4**

In [ ]:
# Calculate the percentiles (2.5th and 97.5th) for each coefficient to construct the CI
lower_percentile = 2.5
upper_percentile = 97.5

# Calculate the lower and upper percentiles for each coefficient
lower_bound = np.percentile(betas, lower_percentile, axis=0)
upper_bound = np.percentile(betas, upper_percentile, axis=0)
ci_length_4 = []
# Print the results
for i, feature in enumerate(['Intercept', 'hotel', 'is_canceled', 'adr']):
    ci_length_4.append(upper_bound[i] - lower_bound[i])
    print(f"{feature}: ({lower_bound[i]:.4f}, {upper_bound[i]:.4f})")

    # Store the confidence interval as a tuple in the list
    ci_list_q_1.append((lower_bound[i], upper_bound[i]))

**Q2**

In [ ]:
ci_lengths = {
    "coefficient": ["Intercept", "hotel", "is_canceled", "adr"],
    "normal_matrix": [],
    "normal_bootstrap": [],
    "pivot_bootstrap": [],
    "quantile_bootstrap": []
}

# Iterate over the coefficients and append the CI lengths
for i in range(len(ci_lengths["coefficient"])):
    # Append the CI length for each method
    ci_lengths["normal_matrix"].append(round(ci_length_1[i], 4))
    ci_lengths["normal_bootstrap"].append(round(ci_length_2[i], 4))
    ci_lengths["pivot_bootstrap"].append(round(ci_length_3[i], 4))
    ci_lengths["quantile_bootstrap"].append(round(ci_length_4[i], 4))

# Create a DataFrame from the dictionary
ci_lengths_df = pd.DataFrame(ci_lengths)

# Set the coefficient as the index
ci_lengths_df = ci_lengths_df.set_index("coefficient", drop=True)

# Display the result
print(ci_lengths_df)


[Embedded submission image omitted from public copy.]

calc original beta (using full data)

In [ ]:
import numpy as np
import pandas as pd
df_sampled_encoded_10000 = df_sampled.copy()
# Map 'hotel' column to 0 and 1
df_sampled_encoded_10000['hotel'] = df_sampled['hotel'].map({'City Hotel': 0, 'Resort Hotel': 1})

# Extracting features and target variable
X_10000 = df_sampled_encoded_10000[['hotel', 'is_canceled', 'adr']].to_numpy()
y_10000 = df_sampled_encoded_10000['lead_time'].to_numpy()

# Adding a column of 1s for the intercept
X_10000 = np.c_[np.ones(X_10000.shape[0]), X_10000]

# Compute beta_hat using the normal equation
beta_hat_10000 = np.linalg.inv(X_10000.T @ X_10000) @ X_10000.T @ y_10000

# Display results
print(f"Intercept (β₀): {beta_hat_10000[0]}")
print(f"hotel Coefficient (β₁): {beta_hat_10000[1]}")
print(f"is canceled Coefficient (β₂): {beta_hat_10000[2]}")
print(f"adr Coefficient (β₃): {beta_hat_10000[3]}")

print ("")

feature_names = ['const', 'hotel', 'is_canceled', 'adr']

# Iterate over each beta and its corresponding confidence intervals in ci_list_q_1
for i in range(len(beta_hat_10000)):  # Iterate through the 4 betas
    for j in range(i, len(ci_list_q_1), 4):  # This grabs 4 CIs for each beta
        lower_bound, upper_bound = ci_list_q_1[j]
        beta_value = beta_hat_10000[i]

        # Check if the beta_value is within the current CI
        is_within_ci = lower_bound <= beta_value <= upper_bound

        # Print the result for the current confidence interval
        print(f"{feature_names[i]},  ({lower_bound:.4f}, {upper_bound:.4f}), Within CI: {is_within_ci}")

[Embedded submission image omitted from public copy.]

**Q 3.1**

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import norm

df_sampled_encoded_10000 = df_sampled.copy()
df_sampled_encoded_9800_unique = df_sampled_encoded_10000[~df_sampled_encoded_10000.isin(df_sampled_encoded_200)].dropna()
df_sampled_encoded_100 = df_sampled_encoded_9800_unique.sample(n=100, random_state=7)
df_sampled_encoded_100['hotel'] = df_sampled_encoded_100['hotel'].map({'City Hotel': 0, 'Resort Hotel': 1})

X_new = df_sampled_encoded_100[['hotel', 'is_canceled', 'adr']]
y_new = df_sampled_encoded_100['lead_time']
X_new = sm.add_constant(X_new)
y_hat = beta_hat_200[0] + beta_hat_200[1]*X_new['hotel'] + beta_hat_200[2]*X_new['is_canceled'] + beta_hat_200[3]*X_new['adr']


**Q 3.2**

In [ ]:

# Bootstrapping
for b in range(B):
    # Generate bootstrap sample
    new_sampled_df = df_sampled_encoded_100.sample(n=n, replace=True)


    # Extract predictors and add constant term
    x_new_sampled = new_sampled_df[['hotel', 'is_canceled', 'adr']].to_numpy()
    x_new_sampled = np.c_[np.ones(x_new_sampled.shape[0]), x_new_sampled]  # Add constant term

    # Compute predicted values
    y_hat_sampled = (
        beta_hat_200[0]
        + beta_hat_200[1] * x_new_sampled[:, 1]  # hotel
        + beta_hat_200[2] * x_new_sampled[:, 2]  # is_canceled
        + beta_hat_200[3] * x_new_sampled[:, 3]  # adr
    )

    # Calculate the lower and upper percentiles for the predictions
    lower_bound = np.percentile(y_hat_sampled, lower_percentile)
    upper_bound = np.percentile(y_hat_sampled, upper_percentile)

    # Append confidence intervals
    ci_y.append([lower_bound, upper_bound])

    # Append the mean of predictions
    bootstrap_y_pred.append(np.mean(y_hat_sampled))


y_new = df_sampled_encoded_100['lead_time'].to_numpy()



**Q 3.3**

In [ ]:
# Check if y_new is within the confidence intervals
in_interval_count = 0

for obs_index, actual_value in enumerate(y_new):
    # Get the confidence interval for the current observation
    ci = ci_y[obs_index]

    # Check if the actual value is within the confidence interval
    if ci[0] <= actual_value <= ci[1]:
        in_interval_count += 1

# Calculate the percentage of y_new values within the intervals
percentage_in_interval = (in_interval_count / len(y_new)) * 100
print(f"Percentage of y_new within confidence intervals: {percentage_in_interval:.2f}%")


[Embedded submission image omitted from public copy.]

**Q 3.4**

[Embedded submission image omitted from public copy.]

## **Part B**

Q1

[Embedded submission image omitted from public copy.]

Q2+Q3a

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.utils import resample

# Load the data (assume it's a CSV file)
df = pd.read_csv("hotel_bookings.csv")

# Select relevant columns and encode categorical variables
selected_columns = ['lead_time', 'is_canceled']
df = df[selected_columns]
df_sample = df[selected_columns].sample(n=200, random_state=26)

df_canceled = df_sample[df_sample['is_canceled'] == 1]  # DataFrame where is_canceled = 1
df_not_canceled = df_sample[df_sample['is_canceled'] == 0]  # DataFrame where is_canceled = 0

print (len(df_canceled))

miu1 = df_canceled['lead_time'].mean()
miu2 = df_not_canceled['lead_time'].mean()
delta = miu1 - miu2

print(f"miu1 = {miu1}")
print(f"miu2 = {miu2}")
print(f"delta = {delta}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot for df_canceled and df_not_canceled
plt.figure(figsize=(12, 6))

# Plot for canceled bookings
plt.subplot(1, 2, 1)
sns.histplot(df_canceled['lead_time'], kde=True, color='red', bins=30)
plt.title('Distribution of Lead Time for Canceled Bookings')
plt.xlabel('Lead Time')
plt.ylabel('Frequency')

# Plot for not canceled bookings
plt.subplot(1, 2, 2)
sns.histplot(df_not_canceled['lead_time'], kde=True, color='green', bins=30)
plt.title('Distribution of Lead Time for Not Canceled Bookings')
plt.xlabel('Lead Time')
plt.ylabel('Frequency')

# Show the plots
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

# Create a figure for the plots
plt.figure(figsize=(12, 6))

# QQ Plot for Canceled bookings
plt.subplot(1, 2, 1)
stats.probplot(df_canceled['lead_time'], dist="norm", plot=plt)
plt.title('QQ Plot for Canceled Bookings')

# QQ Plot for Not Canceled bookings
plt.subplot(1, 2, 2)
stats.probplot(df_not_canceled['lead_time'], dist="norm", plot=plt)
plt.title('QQ Plot for Not Canceled Bookings')

# Show the plots
plt.tight_layout()
plt.show()


[Embedded submission image omitted from public copy.]

Q3b

In [ ]:
# Bootstrap resampling to calculate the confidence interval
B = 400
delta_bootstrap = []

for _ in range(B):
    df_canceled_sample = resample(df_canceled['lead_time'], random_state=_)
    df_not_canceled_sample = resample(df_not_canceled['lead_time'], random_state=_)

    miu1_bootstrap = df_canceled_sample.mean()
    miu2_bootstrap = df_not_canceled_sample.mean()

    delta_bootstrap.append(miu1_bootstrap - miu2_bootstrap)

# Calculate the confidence interval
alpha = 0.05  # 95% confidence interval
lower_bound = np.percentile(delta_bootstrap, 100 * (alpha / 2))
upper_bound = np.percentile(delta_bootstrap, 100 * (1 - alpha / 2))

print(f"95% Confidence Interval for delta: [{lower_bound}, {upper_bound}]")

Q3c

In [ ]:
df_canceled_all = df[df['is_canceled'] == 1]  # DataFrame where is_canceled = 1
df_not_canceled_all = df[df['is_canceled'] == 0]  # DataFrame where is_canceled = 0

miu1_all = df_canceled_all['lead_time'].mean()
miu2_all = df_not_canceled_all['lead_time'].mean()
delta_all = miu1_all - miu2_all

print(f"miu1 = {miu1_all}")
print(f"miu2 = {miu2_all}")
print(f"delta = {delta_all}")
print("Does delta all in the confidence interval?")
if lower_bound <= delta_all <= upper_bound:
    print("Yes")
else:
    print("No")

Q3d

In [ ]:
from scipy.stats import norm

# Group statistics
n1 = df_canceled['lead_time'].size  # Sample size for canceled group
n2 = df_not_canceled['lead_time'].size  # Sample size for not canceled group

var1 = df_canceled['lead_time'].var(ddof=1)  # Variance for canceled group
var2 = df_not_canceled['lead_time'].var(ddof=1)  # Variance for not canceled group

# Standard error of the difference
se_delta = np.sqrt((var1 / n1) + (var2 / n2))

# Wald test statistic
wald_stat = delta / se_delta

# One-sided p-value
p_value_one_sided = 1 - norm.cdf(wald_stat)

# Results
print(f"Observed difference (delta): {delta}")
print(f"Standard error (SE): {se_delta}")
print(f"Wald statistic (W): {wald_stat}")
print(f"One-sided p-value: {p_value_one_sided}")

if p_value_one_sided < 0.05:
    print("Reject the null hypothesis: There is evidence that delta > 0.")
else:
    print("Fail to reject the null hypothesis: Not enough evidence that delta > 0.")


In [ ]:
import numpy as np

# Combine the lead times and is_canceled labels
all_lead_times = df_sample['lead_time'].values
all_labels = df_sample['is_canceled'].values

# Compute the observed difference in means
observed_delta = delta  # Already calculated earlier

# Permutation test
n_permutations = 1000
permuted_deltas = []

for _ in range(n_permutations):
    # Shuffle the labels
    shuffled_labels = np.random.permutation(all_labels)

    # Split the data into two groups based on shuffled labels
    permuted_canceled_mean = all_lead_times[shuffled_labels == 1].mean()
    permuted_not_canceled_mean = all_lead_times[shuffled_labels == 0].mean()

    # Calculate the permuted difference in means
    permuted_deltas.append(permuted_canceled_mean - permuted_not_canceled_mean)

# Calculate the p-value
permuted_deltas = np.array(permuted_deltas)
p_value = np.mean(permuted_deltas >= observed_delta)

# Results
print(f"Observed delta: {observed_delta}")
print(f"P-value: {p_value}")

if p_value < 0.05:
    print("The lead time distribution for canceled bookings is significantly greater than for not canceled bookings.")
else:
    print("No significant difference in lead time distributions.")


Q4a

In [ ]:
m1 = df_canceled['lead_time'].median()
m2 = df_not_canceled['lead_time'].median()
delta_2 = m1 - m2

print(f"miu1 = {m1}")
print(f"miu2 = {m2}")
print(f"delta = {delta_2}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Assuming data_canceled and data_not_canceled are the two datasets
data = df_canceled['lead_time']  # Example for canceled group

# Define the likelihood function (for Laplace distribution)
def log_likelihood(mu, data):
    return -np.sum(np.abs(data - mu))  # Log-likelihood up to a constant

# Test values for mu around the sample median
mu_values = np.linspace(data.median() - 10, data.median() + 10, 100)
likelihoods = [log_likelihood(mu, data) for mu in mu_values]

# Plot the log-likelihood
plt.plot(mu_values, likelihoods, label="Log-Likelihood")
plt.axvline(data.median(), color="red", linestyle="--", label="Sample Median")
plt.title("Log-Likelihood vs. Mu (Canceled Group)")
plt.xlabel("Mu")
plt.ylabel("Log-Likelihood")
plt.legend()
plt.show()

# Check if sample median maximizes the likelihood
sample_median = data.median()
max_likelihood_mu = mu_values[np.argmax(likelihoods)]

print(f"Sample Median: {sample_median}")
print(f"MLE for Mu: {max_likelihood_mu}")
print("The sample median is indeed the MLE as shown in the graph")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Assuming data_canceled and data_not_canceled are the two datasets
data = df_not_canceled['lead_time']  # Example for canceled group

# Define the likelihood function (for Laplace distribution)
def log_likelihood(mu, data):
    return -np.sum(np.abs(data - mu))  # Log-likelihood up to a constant

# Test values for mu around the sample median
mu_values = np.linspace(data.median() - 10, data.median() + 10, 100)
likelihoods = [log_likelihood(mu, data) for mu in mu_values]

# Plot the log-likelihood
plt.plot(mu_values, likelihoods, label="Log-Likelihood")
plt.axvline(data.median(), color="red", linestyle="--", label="Sample Median")
plt.title("Log-Likelihood vs. Mu (Canceled Group)")
plt.xlabel("Mu")
plt.ylabel("Log-Likelihood")
plt.legend()
plt.show()

# Check if sample median maximizes the likelihood
sample_median = data.median()
max_likelihood_mu = mu_values[np.argmax(likelihoods)]

print(f"Sample Median: {sample_median}")
print(f"MLE for Mu: {max_likelihood_mu}")
print("The sample median is indeed the MLE as shown in the graph")


[Embedded submission image omitted from public copy.]

Q4b

In [ ]:
# Bootstrap resampling to calculate the confidence interval for medians
B = 400  # Number of bootstrap iterations
delta_bootstrap = []

for _ in range(B):
    # Resample with replacement
    df_canceled_sample = resample(df_canceled['lead_time'], random_state=_)
    df_not_canceled_sample = resample(df_not_canceled['lead_time'], random_state=_)

    # Calculate medians for bootstrap samples
    m1_bootstrap = np.median(df_canceled_sample)
    m2_bootstrap = np.median(df_not_canceled_sample)

    # Append the difference of medians
    delta_bootstrap.append(m1_bootstrap - m2_bootstrap)

# Calculate the confidence interval
alpha = 0.05  # 95% confidence interval
lower_bound_2 = np.percentile(delta_bootstrap, 100 * (alpha / 2))
upper_bound_2 = np.percentile(delta_bootstrap, 100 * (1 - alpha / 2))

print(f"95% Confidence Interval for delta (Median1 - Median2): [{lower_bound_2}, {upper_bound_2}]")


Q4c

In [ ]:
df_canceled_all = df[df['is_canceled'] == 1]  # DataFrame where is_canceled = 1
df_not_canceled_all = df[df['is_canceled'] == 0]  # DataFrame where is_canceled = 0

m1_all = df_canceled_all['lead_time'].median()
m2_all = df_not_canceled_all['lead_time'].median()
delta_2_all = m1_all - m2_all

print(f"miu1 = {m1_all}")
print(f"miu2 = {m2_all}")
print(f"delta = {delta_2_all}")
print("Does delta all in the confidence interval?")
if lower_bound_2 <= delta_2_all <= upper_bound_2:
    print("Yes")
else:
    print("No")

Q4d

In [ ]:
import numpy as np

# Combine the lead times and is_canceled labels
all_lead_times = df_sample['lead_time'].values
all_labels = df_sample['is_canceled'].values

# Compute the observed difference in median
observed_delta = delta  # Already calculated earlier

# Permutation test
n_permutations = 1000
permuted_deltas = []

for _ in range(n_permutations):
    # Shuffle the labels
    shuffled_labels = np.random.permutation(all_labels)

    # Split the data into two groups based on shuffled labels

    permuted_canceled_median = np.median(all_lead_times[shuffled_labels == 1])
    permuted_not_canceled_median =  np.median(all_lead_times[shuffled_labels == 0])

    # Calculate the permuted difference in median
    permuted_deltas.append(permuted_canceled_median - permuted_not_canceled_median)

# Calculate the p-value
permuted_deltas = np.array(permuted_deltas)
p_value = np.mean(permuted_deltas >= observed_delta)

# Results
print(f"Observed delta: {observed_delta}")
print(f"P-value: {p_value}")

if p_value < 0.05:
    print("The lead time distribution for canceled bookings is significantly greater than for not canceled bookings.")
else:
    print("No significant difference in lead time distributions.")


Q4e

[Embedded submission image omitted from public copy.]

Q5

In [ ]:
import numpy as np
from scipy import stats
import random

# Initialize parameters
B = 1000  # Number of resampling permutations
all_lead_times = df_sample['lead_time'].values
all_labels = df_sample['is_canceled'].values

# Split data into male and female equivalent groups (canceled and not canceled)
num_canceled = np.sum(all_labels == 1)
num_not_canceled = np.sum(all_labels == 0)
n = num_canceled
m = num_not_canceled

# Assign ranks to the lead_times
lead_time_ranks = stats.rankdata(all_lead_times)

# Compute the original test statistic (sum of ranks for canceled bookings)
S1_original = np.sum(lead_time_ranks[all_labels == 1])

# Permutation test
counter = 0
for _ in range(B):
    # Shuffle the labels
    shuffled_labels = np.random.permutation(all_labels)

    # Compute the test statistic for the shuffled labels
    current_S1 = np.sum(lead_time_ranks[shuffled_labels == 1])

    # Increment counter if current test statistic >= original
    if current_S1 >= S1_original:
        counter += 1

# Compute the p-value
p_value = counter / B
print(f"Reject H0 by resampling? {p_value <= 0.05}")


# Check for duplicate rows in the DataFrame
if df_sample.duplicated().any():
    print("There are duplicate rows in df_sample.")
else:
    print("There are no duplicate rows in df_sample.")

# Check for ties
unique, counts = np.unique(all_lead_times, return_counts=True)
ties_exist = np.any(counts > 1)

print(f"Ties exist: {ties_exist}")

# Optional: Display tied values and their counts
tied_values = unique[counts > 1]
tied_counts = counts[counts > 1]

if ties_exist:
    print("Tied values and their counts:")
    for value, count in zip(tied_values, tied_counts):
        print(f"Value: {value}, Count: {count}")
else:
    print("No ties found.")

[Embedded submission image omitted from public copy.]

Q6

[Embedded submission image omitted from public copy.]